# Evaluating deductive and abductive reasoning with Phi-4-mini-reasoning

Phi-4-mini-reasoning is tuned for multi-step reasoning. This notebook checks how well it handles two kinds of **logical** reasoning on small, reproducible benchmarks, and shows where it fails:

| Task | Question | Dataset |
|---|---|---|
| **Deductive** | Given facts and rules, is a statement True or False? | [PARARULE-Plus](https://huggingface.co/datasets/qbao775/PARARULE-Plus) (depth 2-5, MIT) |
| **Abductive** | Given facts, rules and an observation that does not follow, which single missing fact explains it? | [AbductionRules](https://github.com/Strong-AI-Lab/AbductionRules) (Simple split, MIT) |

The code talks to any **OpenAI-compatible endpoint**, so the same notebook runs with Foundry Local, Ollama, vLLM, GitHub Models or a Microsoft Foundry deployment.

In [1]:
%pip install -q openai pandas

## 1. Pick an endpoint

Set `BACKEND` in the next cell. Everything after it is the same for every backend.

* **Foundry Local** (Windows / macOS, runs on your device): `winget install Microsoft.FoundryLocal` or `brew install microsoft/foundrylocal/foundrylocal`, then `pip install foundry-local-sdk`.
* **Ollama**: `ollama pull phi4-mini-reasoning`.
* **vLLM** (Linux GPU): `vllm serve microsoft/Phi-4-mini-reasoning --max-model-len 16384 --served-model-name phi-4-mini-reasoning`.

In [2]:
import os
from openai import OpenAI

BACKEND = "vllm"  # "foundry_local", "ollama" or "vllm" (used for the outputs shown in this notebook)

if BACKEND == "foundry_local":
    from foundry_local import FoundryLocalManager
    alias = "phi-4-mini-reasoning"
    manager = FoundryLocalManager(alias)
    client = OpenAI(base_url=manager.endpoint, api_key=manager.api_key)
    MODEL = manager.get_model_info(alias).id
elif BACKEND == "ollama":
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "phi4-mini-reasoning"
else:  # vLLM or any other OpenAI-compatible server
    client = OpenAI(base_url=os.getenv("PHI_BASE_URL", "http://localhost:8000/v1"), api_key="not-needed")
    MODEL = "phi-4-mini-reasoning"

# Sampling settings recommended on the Phi-4-mini-reasoning model card
GEN = dict(temperature=0.8, top_p=0.95, max_tokens=12288)

## 2. Load the benchmarks

`reasoning_eval.py` (next to this notebook) downloads the two test sets once, caches them in `./data`, and draws a fixed-seed sample:

* **Deductive**: equal numbers of True and False statements at every reasoning depth (2-5 rule applications). Half of the items use rules with negation (`If someone is not big then ...`), which are read under the closed-world assumption.
* **Abductive**: one positive observation per context from the *Simple* split, where exactly one missing fact explains the observation, so the answer can be scored by exact match.

The defaults below give 80 items so the notebook finishes in a few minutes. Increase them for tighter estimates.

In [3]:
import reasoning_eval as R

deductive = R.load_deductive(per_depth=10)     # 4 depths x 10 = 40 items
abductive = R.load_abductive(per_domain=20)    # Animal + Person = 40 items
examples = deductive + abductive
print(len(deductive), "deductive,", len(abductive), "abductive")

ex = abductive[0]
print("\nCONTEXT:", ex.context[:400], "...")
print("OBSERVATION:", ex.question)
print("GOLD MISSING FACT:", ex.answer)

40 deductive, 40 abductive

CONTEXT: If an animal visits the dog, attacks the cat, and is obese, then it is reckless. The bear is lazy. The cat is smart. If something is dull, chases the cat, and likes the dog, then it is heavy. The cat is kind. The bear is tired. If something is quiet, is kind, and is nice, then it is also round. The cat is nice. The leopard is strong. If an animal is furry, is funny, and is adorable, then it is sma ...
OBSERVATION: The bear is rough.
GOLD MISSING FACT: The bear is boring.


## 3. Look at one prompt and one answer

In [4]:
print(R.build_prompt(ex)[-450:])
[r] = R.run_eval(client, MODEL, [ex], **GEN)
print("\n--- model output (last 400 chars) ---\n", r.raw[-400:])
print("\nprediction:", r.prediction, "| correct:", r.correct, "| tokens:", r.completion_tokens)

 is fierce, is strong, and is big, then it is awful. The dog is cute. The bear chases the cat. The dog is furry. The leopard visits the dog. The leopard is fierce. The bear is dull. If an animal is tired, is lazy, and is boring, it is rough. The dog is adorable.

Observation: The bear is rough.

Reason step by step, then finish with a single line of the form
Missing fact: <one short sentence in the same style as the facts, e.g. "The cat is big.">



--- model output (last 400 chars) ---
 le is rule 6, which requires tired, lazy, and boring. The bear is already tired and lazy. So the missing fact is "The bear is boring."

Alternatively, perhaps there's a different rule. Let me check all rules again:

Rule 6: tired, lazy, boring → rough. That's the only one that can make the bear rough.

Therefore, the missing fact is "The bear is boring."
</think>

Missing fact: The bear is boring.

prediction: The bear is boring. | correct: True | tokens: 2050


## 4. Run the evaluation

Requests are sent in parallel. Each answer is parsed from the text after the `</think>` block:

* deductive: the last `Final answer: True/False`
* abductive: the last `Missing fact: ...`, compared with the gold fact after lower-casing and stripping punctuation

In [5]:
import time
t0 = time.time()
results = R.run_eval(client, MODEL, examples, workers=32, **GEN)
print(f"done in {time.time() - t0:.0f}s")
R.save_results(results, "results_notebook.jsonl")

done in 218s


## 5. Accuracy by task and slice

In [6]:
import pandas as pd

summary = R.summarize(results)
display(pd.DataFrame(summary["accuracy"]).T)
print("truncated responses:", summary["truncated"], "| mean completion tokens:", summary["mean_completion_tokens"])

,n,accuracy
abductive,40.0,0.850
abductive/Animal,20.0,0.800
abductive/Person,20.0,0.900
deductive,40.0,0.700
deductive/depth=2,10.0,0.900
deductive/depth=3,10.0,0.800
deductive/depth=4,10.0,0.500
deductive/depth=5,10.0,0.600
deductive/negation,19.0,0.526
deductive/no-negation,21.0,0.857


truncated responses: 0 | mean completion tokens: 2697


## 6. Error analysis

Error types:

| Task | Error type | Meaning |
|---|---|---|
| deductive | `false_positive` / `false_negative` | answered True for a False statement / False for a True one |
| abductive | `wrong_attribute` | right entity, wrong property (often a fact that a *different* rule would need) |
| abductive | `wrong_entity` | proposed a fact about a different entity |
| abductive | `restated_known_fact` | "explained" the observation with a fact already in the context |
| both | `truncated` / `no_answer` | hit `max_tokens` inside `<think>`, or no parsable final line |

In [7]:
display(pd.DataFrame(summary["errors"]).fillna(0).astype(int))

df = pd.DataFrame([{
    "task": r.example.task, "question": r.example.question, "gold": r.example.answer,
    "prediction": r.prediction, "error": r.error_type, "tokens": r.completion_tokens,
} for r in results])
display(df[df.error.notna()].head(10))

,deductive,abductive
false_positive,6,0
false_negative,6,0
wrong_attribute,0,5
restated_known_fact,0,1


,task,question,gold,prediction,error,tokens
4,deductive,The dog is not smart.,False,True,false_positive,3750
6,deductive,The rabbit is rough.,True,False,false_negative,2401
14,deductive,Gary is not wealthy.,False,True,false_positive,2978
18,deductive,Alan is not famous.,False,True,false_positive,2742
22,deductive,The cat is funny.,True,False,false_negative,3925
24,deductive,Alan is wealthy.,True,False,false_negative,5715
25,deductive,The bald eagle is not awful.,False,True,false_positive,8152
27,deductive,Alan is not quiet.,False,True,false_positive,3716
29,deductive,Fiona is dull.,True,False,false_negative,4066
34,deductive,The tiger is big.,True,False,false_negative,7989


Token budget also tells you something: reasoning models tend to think longer when they are about to fail.

In [8]:
display(df.assign(correct=df.error.isna()).groupby(["task", "correct"]).tokens.describe()[["count", "mean", "50%", "max"]])

count         mean     50%     max
task      correct                                    
abductive False      6.0  4245.333333  4450.5  7676.0
          True      34.0  2063.882353  1497.5  7861.0
deductive False     12.0  4397.000000  3837.5  8152.0
          True      28.0  2406.071429  1606.5  8585.0

## 7. Takeaways and next steps

On the full 400-item run (see the [README](./README.md) for the complete table, 3 runs):

* **Abduction**: Phi-4-mini-reasoning reaches 88% exact match, 24 points above Phi-4-mini-instruct. The instruct model often answers `None`, denying that any fact is missing.
* **Deduction**: about 89% on rules without negation, but about 56% on rules with negation. Reading `not X` under the closed-world assumption is the main weakness, so report this slice separately rather than relying on one aggregate number.
* **Token count is a confidence signal**: wrong answers use 2.6-4.2x more completion tokens (median) than correct ones. Long traces are good candidates for a retry or a human check.
* Check truncation before blaming the model: if responses end inside `<think>`, raise `max_tokens`.

To compare models on the same items, change `MODEL` to `Phi-4-mini-instruct`, `Phi-4-reasoning` or your own fine-tuned checkpoint. `python reasoning_eval.py --help` runs the same evaluation from the command line.

### References

* Bao et al., 2022. *Multi-Step Deductive Reasoning Over Natural Language: An Empirical Study on Out-of-Distribution Generalisation.* IJCLR-NeSy 2022. (PARARULE-Plus)
* Young, Bao, Bensemann and Witbrock, 2022. *AbductionRules: Training Transformers to Explain Unexpected Inputs.* Findings of ACL 2022.
* [Phi-4-mini-reasoning technical report](https://huggingface.co/microsoft/Phi-4-mini-reasoning/blob/main/Phi-4-Mini-Reasoning.pdf)